# Bronze Layer v2 — Hourly Scheduled Load: Source Blob Realtime → All Entities

**Day 6 | Scheduled via Databricks Job — runs every hour automatically.**

Reads the current hour's CSV files from the instructor-provided source blob for **all realtime entities** and copies them into the Bronze Volume, preserving the exact source directory structure. No transformation — files land in Bronze exactly as they are in the source.

Same pattern as `02_bronze_blob_charging_sessions_v2.ipynb` from Day 3 but covers all realtime entities in one scheduled job.

---

### Source layout (instructor blob)

```
wasbs://source@dataenggdailystorage.blob.core.windows.net/
  └── realtime/
        ├── charging_sessions/      YYYY/MM/DD/HH/  sessions_YYYYMMDD_HHMM.csv
        │                                           charging_sessions_YYYYMMDD_HHMM.csv  ← some hours
        └── maintenance_events/     YYYY/MM/DD/HH/  maintenance_YYYYMMDD_HHMM.csv
                                                    maintenance_events_YYYYMMDD_HHMM.csv ← some hours
```

> Only 2 entities in `realtime/`. Other EV data comes from the VoltGrid API (Day 5 ADF pipeline).

### Bronze Volume target (mirrors source exactly)

```
/Volumes/dbw_ev_intelligence_dev/default/bronze-volume/
  └── realtime/
        ├── charging_sessions/      YYYY/MM/DD/HH/  *.csv
        └── maintenance_events/     YYYY/MM/DD/HH/  *.csv
```

---

### What changed from v1 → v2

| | v1 | v2 |
|---|---|---|
| **Trigger** | Manual | Databricks Job — cron `0 * * * *` (top of every hour) |
| **Partition** | You set `LOAD_YEAR/MONTH/DAY/HOUR` manually | Auto-computed from `datetime.now(UTC)` |
| **Full load** | `LOAD_MODE = "full"` | `FULL_LOAD_OVERRIDE = True` (one-off flag) |
| **Look-back** | Single hour | 3-hour window — catches late-arriving data |
| **Already loaded** | Overwrites | Checks Bronze first — skips hours already present |
| **Source missing** | Crashes | Checks source first — exits cleanly if folder absent |
| **Multi-entity** | One entity per notebook | All realtime entities in one run |

## Cell 1 — Authenticate to source blob storage

Reads storage account, container, and SAS token from Key Vault via `kv-ev-scope` Databricks secret scope.

**Must run every session** — Spark config does not persist across cluster restarts.

In [ ]:
from datetime import datetime, timezone, timedelta

SCOPE = "kv-ev-scope"

STORAGE_ACCOUNT = dbutils.secrets.get(scope=SCOPE, key="source-storage-account")
CONTAINER       = dbutils.secrets.get(scope=SCOPE, key="source-container")
SAS_TOKEN       = dbutils.secrets.get(scope=SCOPE, key="source-sas-token")

spark.conf.set(
    f"fs.azure.sas.{CONTAINER}.{STORAGE_ACCOUNT}.blob.core.windows.net",
    SAS_TOKEN
)

SOURCE_ROOT = f"wasbs://{CONTAINER}@{STORAGE_ACCOUNT}.blob.core.windows.net"

print(f"Storage account : {STORAGE_ACCOUNT}")
print(f"Container       : {CONTAINER}")
print(f"SAS token       : [REDACTED]")
print(f"Source root     : {SOURCE_ROOT}")
print("Source blob authenticated — OK")

## Cell 2 — Read `load_type` Job parameter and build window

`load_type` is passed by the Databricks Job as a widget parameter.

| `load_type` | Behaviour |
|---|---|
| `full` | Copies all historical files across all entities — use for first run |
| `incremental` | Auto 3-hour look-back from `datetime.now(UTC)` — use for all scheduled runs |

When running interactively (not from a Job), the widget falls back to `"incremental"` if not set.

In [ ]:
from datetime import datetime, timezone, timedelta

# Read load_type from Databricks Job widget parameter.
# Default: "incremental" so the notebook is safe to run manually without setting the widget.
dbutils.widgets.text("load_type", "incremental", "Load Type (full / incremental)")
load_type = dbutils.widgets.get("load_type").strip().lower()

if load_type not in ("full", "incremental"):
    raise ValueError(f"Invalid load_type='{load_type}'. Must be 'full' or 'incremental'.")

print(f"load_type       : {load_type}")

LOOKBACK_HOURS = 3

# Realtime entity folders in the source blob under realtime/
# Only 2 entities — other EV data comes from VoltGrid API (Day 5 ADF)
ENTITIES = [
    "charging_sessions",
    "maintenance_events",
]

BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"
BASE_SUBPATH  = "realtime"

now = datetime.now(timezone.utc)
print(f"Job fire time (UTC) : {now.strftime('%Y-%m-%d %H:%M:%S')} UTC")

if load_type == "full":
    # Full load — copy entire history for every entity, skip-if-already-in-Bronze still applies
    window = []
    for entity in ENTITIES:
        window.append({
            "entity":      entity,
            "source_path": f"{SOURCE_ROOT}/{BASE_SUBPATH}/{entity}/",
            "bronze_path": f"{BRONZE_VOLUME}/{BASE_SUBPATH}/{entity}/",
            "label":       f"{entity} — FULL"
        })
else:
    # Incremental — 3-hour look-back window per entity
    # e.g. job fires 09:00 UTC → checks hours 08, 07, 06 for each entity
    window = []
    for offset in range(1, LOOKBACK_HOURS + 1):
        target    = now - timedelta(hours=offset)
        partition = target.strftime("%Y/%m/%d/%H")
        for entity in ENTITIES:
            window.append({
                "entity":      entity,
                "source_path": f"{SOURCE_ROOT}/{BASE_SUBPATH}/{entity}/{partition}/",
                "bronze_path": f"{BRONZE_VOLUME}/{BASE_SUBPATH}/{entity}/{partition}/",
                "label":       f"{entity} — {partition}"
            })

print(f"\nWindow ({len(window)} slot(s)):")
for w in window:
    print(f"  {w['label']}")

## Cell 3 — Filter window: skip already loaded, skip source missing

For each entity-hour slot in the window, two checks run before any copy:

**Check 1 — Already in Bronze?**
If the Bronze destination folder exists and has at least one file, that entity-hour was already loaded. Skip it — no duplicate copy needed.

**Check 2 — Source folder exists?**
If the source blob has no folder for that entity-hour, data has not arrived yet. Skip and move on — the next run will still include this slot if it falls within 3 hours.

Only slots passing both checks proceed to the copy step.

```
Example — job fires 09:00 UTC, entity = charging_sessions:
  2026/07/11/08 → Bronze has data  → SKIP (already loaded)
  2026/07/11/07 → Source missing   → SKIP (not arrived yet)
  2026/07/11/06 → Both exist       → COPY
```

In [ ]:
def folder_exists(path):
    try:
        files = dbutils.fs.ls(path)
        return len(files) > 0
    except Exception:
        return False

slots_to_copy = []

for w in window:
    label       = w["label"]
    source_path = w["source_path"]
    bronze_path = w["bronze_path"]

    # Incremental only: skip hours already loaded in Bronze
    if load_type == "incremental" and folder_exists(bronze_path):
        print(f"  SKIP (already in Bronze) : {label}")
        continue

    # Both modes: skip if source folder does not exist
    if not folder_exists(source_path):
        print(f"  SKIP (source not found)  : {label}")
        continue

    print(f"  QUEUE for copy           : {label}")
    slots_to_copy.append(w)

print(f"\nSlots queued for copy: {len(slots_to_copy)}")

if not slots_to_copy:
    msg = f"Nothing to copy — load_type={load_type}, all slots already loaded or source not found."
    print(f"\nINFO: {msg}")
    dbutils.notebook.exit(msg)

## Cell 4 — List source files for each queued slot

Recursively lists all files under each queued entity-hour source path. For a normal scheduled run each slot typically has one CSV file.

In [ ]:
def list_files_recursive(path):
    try:
        items = dbutils.fs.ls(path)
    except Exception:
        return []
    files = []
    for item in items:
        if item.isDir():
            files.extend(list_files_recursive(item.path))
        else:
            files.append(item)
    return files

for w in slots_to_copy:
    w["source_files"] = list_files_recursive(w["source_path"])
    print(f"  {w['label']} — {len(w['source_files'])} file(s)")
    for f in w["source_files"]:
        print(f"    {f.path.replace(w['source_path'], '')}  [{round(f.size/1024, 1)} KB]")

## Cell 5 — Copy files to Bronze Volume

Copies files from source to Bronze for every queued entity-hour slot.

**How destination path is built:**
```
source file  : wasbs://.../realtime/charging_sessions/2026/07/11/07/sessions_20260711_0700.csv
source_path  : wasbs://.../realtime/charging_sessions/2026/07/11/07/
relative     : sessions_20260711_0700.csv
dest_path    : /Volumes/.../realtime/charging_sessions/2026/07/11/07/sessions_20260711_0700.csv
```

The `YYYY/MM/DD/HH/` folder hierarchy on Bronze is created automatically by `dbutils.fs.cp`.
On failure, an exception is raised — Job marks run Failed and sends alert.

In [ ]:
all_copied  = []
all_skipped = []

for w in slots_to_copy:
    source_path  = w["source_path"]
    bronze_path  = w["bronze_path"]
    source_files = w["source_files"]
    label        = w["label"]

    copied  = []
    skipped = []

    print(f"\n--- {label} ---")
    for file_info in source_files:
        relative_path = file_info.path.replace(source_path, "")
        dest_path     = bronze_path + relative_path
        try:
            dbutils.fs.cp(file_info.path, dest_path)
            copied.append(dest_path)
            print(f"  COPIED  {relative_path}")
        except Exception as e:
            skipped.append((file_info.path, str(e)))
            print(f"  FAILED  {relative_path} — {e}")

    print(f"  Result: {len(copied)} copied, {len(skipped)} failed")
    all_copied.extend(copied)
    all_skipped.extend(skipped)

print(f"\nTotal: {len(all_copied)} copied, {len(all_skipped)} failed across {len(slots_to_copy)} slot(s)")

if all_skipped:
    raise Exception(f"{len(all_skipped)} file(s) failed to copy — check output above.")

## Cell 6 — Verify files in Bronze Volume per slot

For each slot that was copied, lists files in the Bronze destination and asserts count matches source. If any assertion fails the Job run is marked Failed and an alert fires.

In [ ]:
for w in slots_to_copy:
    bronze_path  = w["bronze_path"]
    source_files = w["source_files"]
    label        = w["label"]

    bronze_files = list_files_recursive(bronze_path)
    status = "PASS" if len(bronze_files) == len(source_files) else "FAIL"
    print(f"  [{status}] {label} — source={len(source_files)}  bronze={len(bronze_files)}")

    assert len(bronze_files) == len(source_files), (
        f"Count mismatch for {label} — source: {len(source_files)}, bronze: {len(bronze_files)}"
    )

print("\nAll slots verified.")

## Cell 7 — Read sample file from first copied slot and inspect schema

Reads the first CSV from the first copied entity-hour into Spark — lightweight sanity check to confirm the file is readable and columns look as expected.

In [ ]:
first_slot   = slots_to_copy[0]
bronze_files = list_files_recursive(first_slot["bronze_path"])

if bronze_files:
    sample_file = bronze_files[0].path
    print(f"Reading sample from: {first_slot['label']}")
    print(f"File: {sample_file.replace(first_slot['bronze_path'], '')}")

    df = spark.read \
        .option("header", True) \
        .option("inferSchema", True) \
        .csv(sample_file)

    print(f"Row count : {df.count():,}")
    print(f"Columns   : {len(df.columns)}")
    df.printSchema()
    display(df.limit(5))
else:
    print("No files to sample.")

## Cell 8 — Run summary

Prints a final summary of the run. Visible in **Databricks → Workflows → `job_bronze_all_entities_hourly` → Run history → this run → Output**.

---

**What comes next:** This notebook lands raw CSVs in the Bronze Volume. The Silver layer notebook reads from `/Volumes/.../bronze-volume/realtime/<entity>/`, applies explicit schema, deduplicates, and writes Delta.

In [ ]:
print("=" * 60)
print("BRONZE BLOB REALTIME MIGRATION — RUN SUMMARY")
print("=" * 60)
print(f"load_type           : {load_type}")
print(f"Job fire time (UTC) : {now.strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"Look-back window    : {LOOKBACK_HOURS} hour(s)  [incremental only]")
print(f"Entities            : {len(ENTITIES)}")
print(f"Total slots         : {len(window)}")
print(f"Slots copied        : {len(slots_to_copy)}")
print(f"Slots skipped       : {len(window) - len(slots_to_copy)}")
print(f"Total files copied  : {len(all_copied)}")
print(f"Total files failed  : {len(all_skipped)}")
print()
for w in window:
    status = "COPIED" if w in slots_to_copy else "SKIPPED"
    print(f"  [{status}] {w['label']}")
print("=" * 60)
print("Next step: Silver layer reads from Bronze Volume and writes Delta.")